In [1]:
# ============================================================
#   Role-Based School Management System
#   Python | OOP | Role-Based Access Control
# ============================================================

# ──────────────────────────────────────────────
# DATA STORE  (acts as our in-memory "database")
# ──────────────────────────────────────────────

students = [
    {"id": 101, "name": "RAVINDRA BAIS",   "class": 9,  "marks": 85},
    {"id": 102, "name": "JAGRITI CHOUDARY",    "class": 9,  "marks": 90},
    {"id": 103, "name": "SAURABH GUPTA",     "class": 9,  "marks": 78},
    {"id": 104, "name": "aram kserwani",     "class": 9,  "mark": 77},
    {"id": 105, "name": "MUSKHAN PRAJAPATI",    "class": 10, "marks": 92},
    {"id": 106, "name": "VIKHAS SHUKLA",    "class": 10, "marks": 76},
    {"id": 107, "name": "SHIVAM PANDEY",    "class": 10, "marks": 88},
    {"id": 108, "name": "alooo singh",      "class": 10, "marks": 62},
]

# next_id counter so new students get unique IDs
_next_student_id = 107

users = [
    {
        "email":    "principal@school.com",
        "password": "admin123",
        "name":     "Dr. A.PANDAY",
        "role":     "Principal",
        "class":    None,          # Principal has no single class
    },
    {
        "email":    "teacher9@school.com",
        "password": "teach9pass",
        "name":     "Mrs. Diya Rao",
        "role":     "Teacher",
        "class":    9,
    },
    {
        "email":    "teacher10@school.com",
        "password": "teach10pass",
        "name":     "Mr. Vikram Joshi",
        "role":     "Teacher",
        "class":    10,
    },
]


# ──────────────────────────────────────────────
# HELPER UTILITIES
# ──────────────────────────────────────────────

def separator(char="─", width=55):
    print(char * width)

def header(title):
    separator("═")
    print(f"  {title}")
    separator("═")

def pause():
    input("\n  Press Enter to continue...")

def print_students(student_list):
    if not student_list:
        print("  No students found.")
        return
    separator()
    print(f"  {'ID':<6} {'Name':<22} {'Class':<8} {'Marks'}")
    separator()
    for s in student_list:
        print(f"  {s['id']:<6} {s['name']:<22} {s['class']:<8} {s['marks']}")
    separator()
    print(f"  Total: {len(student_list)} student(s)")


# ──────────────────────────────────────────────
# CLASSES
# ──────────────────────────────────────────────

class User:
    """Base class for all users."""

    def __init__(self, email, name, role, assigned_class):
        self.email          = email
        self.name           = name
        self.role           = role
        self.assigned_class = assigned_class   # None for Principal

    def __str__(self):
        cls = f"Class {self.assigned_class}" if self.assigned_class else "All Classes"
        return f"{self.name}  [{self.role} – {cls}]"

    def logout(self):
        print(f"\n GOODBYE SIR/MAM HAVE A NICE DAY, {self.name}! Logged out successfully.")
        separator()


# ──────── PRINCIPAL ────────

class Principal(User):

    def __init__(self, email, name):
        super().__init__(email, name, "Principal", None)

    def dashboard(self):
        while True:
            header(f"PRINCIPAL DASHBOARD  |  {self.name}")
            print("  1. View All Students")
            print("  2. View Class 9 Students")
            print("  3. View Class 10 Students")
            print("  4. Add New Student")
            print("  5. Add New Teacher")
            print("  6. Assign Teacher to Class")
            print("  7. Update Student Details")
            print("  8. Logout")
            separator()
            choice = input("  Enter choice: ").strip()

            if   choice == "1": self.view_students()
            elif choice == "2": self.view_students(class_filter=9)
            elif choice == "3": self.view_students(class_filter=10)
            elif choice == "4": self.add_student()
            elif choice == "5": self.add_teacher()
            elif choice == "6": self.assign_teacher()
            elif choice == "7": self.update_student()
            elif choice == "8":
                self.logout()
                break
            else:
                print("  ⚠  Invalid choice. Please try again.")
                pause()

    # ---- view ----
    def view_students(self, class_filter=None):
        title = f"ALL STUDENTS" if class_filter is None else f"CLASS {class_filter} STUDENTS"
        header(title)
        data = students if class_filter is None else [s for s in students if s["class"] == class_filter]
        print_students(data)
        pause()

    # ---- add student ----
    def add_student(self):
        global _next_student_id
        header("ADD NEW STUDENT")
        name = input("  Student Name : ").strip()
        if not name:
            print("  ⚠  Name cannot be empty.")
            pause()
            return
        cls = input("  Class (9/10) : ").strip()
        if cls not in ("9", "10"):
            print("  ⚠  Class must be 9 or 10.")
            pause()
            return
        marks_raw = input("  Marks (0-100): ").strip()
        if not marks_raw.isdigit() or not (0 <= int(marks_raw) <= 100):
            print("  ⚠  Marks must be a number between 0 and 100.")
            pause()
            return
        student = {"id": _next_student_id, "name": name, "class": int(cls), "marks": int(marks_raw)}
        students.append(student)
        _next_student_id += 1
        print(f"\n  ✔  Student '{name}' added successfully! (ID: {student['id']})")
        pause()

    # ---- add teacher ----
    def add_teacher(self):
        header("ADD NEW TEACHER")
        name  = input("  Teacher Name    : ").strip()
        email = input("  Email ID        : ").strip()
        pwd   = input("  Password        : ").strip()
        cls   = input("  Assigned Class (9/10): ").strip()

        if not all([name, email, pwd]):
            print("  ⚠  All fields are required.")
            pause()
            return
        if cls not in ("9", "10"):
            print("  ⚠  Class must be 9 or 10.")
            pause()
            return
        if any(u["email"] == email for u in users):
            print("  ⚠  A user with this email already exists.")
            pause()
            return

        users.append({"email": email, "password": pwd, "name": name, "role": "Teacher", "class": int(cls)})
        print(f"\n  ✔  Teacher '{name}' added for Class {cls}.")
        pause()

    # ---- assign teacher ----
    def assign_teacher(self):
        header("ASSIGN TEACHER TO CLASS")
        teachers = [u for u in users if u["role"] == "Teacher"]
        if not teachers:
            print("  No teachers in the system.")
            pause()
            return
        separator()
        for i, t in enumerate(teachers, 1):
            print(f"  {i}. {t['name']}  (Current Class: {t['class']})")
        separator()
        idx = input("  Select teacher number: ").strip()
        if not idx.isdigit() or not (1 <= int(idx) <= len(teachers)):
            print("  ⚠  Invalid selection.")
            pause()
            return
        teacher = teachers[int(idx) - 1]
        new_cls = input("  New Class (9/10): ").strip()
        if new_cls not in ("9", "10"):
            print("  ⚠  Class must be 9 or 10.")
            pause()
            return
        # Update in users list
        for u in users:
            if u["email"] == teacher["email"]:
                u["class"] = int(new_cls)
                break
        print(f"\n  ✔  {teacher['name']} reassigned to Class {new_cls}.")
        pause()

    # ---- update student ----
    def update_student(self):
        header("UPDATE STUDENT DETAILS")
        sid = input("  Enter Student ID: ").strip()
        if not sid.isdigit():
            print("  ⚠  Invalid ID.")
            pause()
            return
        student = next((s for s in students if s["id"] == int(sid)), None)
        if not student:
            print("  ⚠  Student not found.")
            pause()
            return
        print(f"\n  Current → ID: {student['id']} | Name: {student['name']} | Class: {student['class']} | Marks: {student['marks']}")
        separator()
        print("  Leave blank to keep current value.")
        name   = input(f"  New Name  [{student['name']}] : ").strip()
        cls    = input(f"  New Class [{student['class']}] (9/10): ").strip()
        marks  = input(f"  New Marks [{student['marks']}] (0-100): ").strip()

        if name:
            student["name"] = name
        if cls:
            if cls not in ("9", "10"):
                print("  ⚠  Class must be 9 or 10. Class not updated.")
            else:
                student["class"] = int(cls)
        if marks:
            if not marks.isdigit() or not (0 <= int(marks) <= 100):
                print("  ⚠  Marks invalid. Marks not updated.")
            else:
                student["marks"] = int(marks)

        print(f"\n  ✔  Student ID {student['id']} updated successfully.")
        pause()


# ──────── TEACHER ────────

class Teacher(User):

    def __init__(self, email, name, assigned_class):
        super().__init__(email, name, "Teacher", assigned_class)

    def _my_students(self):
        return [s for s in students if s["class"] == self.assigned_class]

    def dashboard(self):
        while True:
            header(f"TEACHER DASHBOARD  |  {self.name}  (Class {self.assigned_class})")
            print("  1. View My Students")
            print("  2. Add Student Marks")
            print("  3. Update Student Marks")
            print("  4. Logout")
            separator()
            choice = input("  Enter choice: ").strip()

            if   choice == "1": self.view_students()
            elif choice == "2": self.add_marks()
            elif choice == "3": self.update_marks()
            elif choice == "4":
                self.logout()
                break
            else:
                print("  ⚠  Invalid choice. Please try again.")
                pause()

    def view_students(self, class_requested=None):
        """View students. If class_requested differs from assigned class → access denied."""
        if class_requested is not None and class_requested != self.assigned_class:
            print(f"\n  ✖  Access Denied – You are not authorized to access this class.")
            pause()
            return
        header(f"CLASS {self.assigned_class} – STUDENT LIST")
        print_students(self._my_students())
        pause()

    def _find_my_student(self, sid):
        student = next((s for s in students if s["id"] == int(sid)), None)
        if not student:
            print("  ⚠  Student not found.")
            return None
        if student["class"] != self.assigned_class:
            print(f"\n  ✖  Access Denied – You are not authorized to access this class.")
            return None
        return student

    def add_marks(self):
        header("ADD STUDENT MARKS")
        sid = input("  Enter Student ID: ").strip()
        if not sid.isdigit():
            print("  ⚠  Invalid ID.")
            pause()
            return
        student = self._find_my_student(sid)
        if not student:
            pause()
            return
        marks = input(f"  Current Marks: {student['marks']}  |  New Marks (0-100): ").strip()
        if not marks.isdigit() or not (0 <= int(marks) <= 100):
            print("  ⚠  Marks must be 0–100.")
            pause()
            return
        student["marks"] = int(marks)
        print(f"\n  ✔  Marks for {student['name']} set to {student['marks']}.")
        pause()

    def update_marks(self):
        """Alias – identical flow to add_marks for this system."""
        self.add_marks()


# ──────────────────────────────────────────────
# LOGIN MODULE
# ──────────────────────────────────────────────

class LoginSystem:

    def authenticate(self, email, password):
        for u in users:
            if u["email"] == email and u["password"] == password:
                return u
        return None

    def run(self):
        header("SCHOOL MANAGEMENT SYSTEM  –  LOGIN")
        print("  Default accounts:")
        print("    principal@school.com  /  admin123")
        print("    teacher9@school.com   /  teach9pass")
        print("    teacher10@school.com  /  teach10pass")
        separator()

        for attempt in range(1, 4):
            print(f"\n  Login Attempt {attempt} of 3")
            email = input("  Email    : ").strip()
            pwd   = input("  Password : ").strip()

            user_data = self.authenticate(email, pwd)
            if user_data:
                print(f"\n  ✔  Welcome, {user_data['name']}! Login successful.\n")
                return user_data
            else:
                print("  ✖  Invalid Credentials. Please try again.")

        print("\n  Too many failed attempts. Exiting system.")
        return None


# ──────────────────────────────────────────────
# FACTORY – create the right user object
# ──────────────────────────────────────────────

def create_user_object(user_data):
    if user_data["role"] == "Principal":
        return Principal(user_data["email"], user_data["name"])
    else:
        return Teacher(user_data["email"], user_data["name"], user_data["class"])


# ──────────────────────────────────────────────
# MAIN ENTRY POINT
# ──────────────────────────────────────────────

def main():
    login_system = LoginSystem()

    while True:
        user_data = login_system.run()
        if not user_data:
            break

        user_obj = create_user_object(user_data)
        user_obj.dashboard()

        again = input("\n  Return to login screen? (y/n): ").strip().lower()
        if again != "y":
            print("\n  Thank you for using School Management System. Goodbye!\n")
            break


if __name__ == "__main__":
    main()

═══════════════════════════════════════════════════════
  SCHOOL MANAGEMENT SYSTEM  –  LOGIN
═══════════════════════════════════════════════════════
  Default accounts:
    principal@school.com  /  admin123
    teacher9@school.com   /  teach9pass
    teacher10@school.com  /  teach10pass
───────────────────────────────────────────────────────

  Login Attempt 1 of 3


  Email    :  principal@school.com
  Password :  admin123



  ✔  Welcome, Dr. A.PANDAY! Login successful.

═══════════════════════════════════════════════════════
  PRINCIPAL DASHBOARD  |  Dr. A.PANDAY
═══════════════════════════════════════════════════════
  1. View All Students
  2. View Class 9 Students
  3. View Class 10 Students
  4. Add New Student
  5. Add New Teacher
  6. Assign Teacher to Class
  7. Update Student Details
  8. Logout
───────────────────────────────────────────────────────


  Enter choice:  8



 GOODBYE SIR/MAM HAVE A NICE DAY, Dr. A.PANDAY! Logged out successfully.
───────────────────────────────────────────────────────



  Return to login screen? (y/n):  yes



  Thank you for using School Management System. Goodbye!

